<center><h1>Leonez, Teresa HW 7</h1></center>

Name: Teresa Leonez
<br>
Github Username: ttl229
<br>
USC ID: 

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [36]:
# Import general packages
import seaborn as sns
import numpy as np
import pandas as pd

# Import model packages
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score

In [48]:
# Helper functions
def get_exact_match(y_true, y_pred):
# Returns the exact match ratio as between 0 and 1 from predicted vs actual
    return np.mean(np.all(np.array(y_true) == np.array(y_pred), axis=1))

def hamming_loss(y_true, y_pred):
# Returns the error rate how many labels were misclassified from pred vs actual
    return np.mean(np.array(y_true) != np.array(y_pred))

def hamming_score(y_true, y_pred):
# Returns share of labels accurately predicted from pred vs actual
    return 1 - hamming_loss(y_true, y_pred)

# Function for Question 1.b..ii.
def train_svm_by_label(model, params, X_train, X_test, y_train, y_test):
    preds, rows = {}, []
    for label in y_train.columns:
        # Use 10-fold CV
        grid = GridSearchCV(model, params, cv=10, n_jobs=-1, verbose=1)
        grid.fit(X_train, y_train[label])
        preds[label] = grid.predict(X_test)
        row = {'Label': label, 'CV Accuracy': grid.best_score_}
        row.update({k.replace('estimator__', ''): v for k, v in grid.best_params_.items()})
        rows.append(row)

    y_pred = pd.DataFrame(preds, index=y_test.index)
    params_df = pd.DataFrame(rows)
    scores_df = pd.DataFrame({
        'Exact Match': [get_exact_match(y_test, y_pred)],
        'Hamming Loss': [hamming_loss(y_test, y_pred)],
        'Hamming Score': [hamming_score(y_test, y_pred)]
    })
    display(params_df)
    display(scores_df)
    return y_pred, scores_df

# Function to choose best K using CH score
def pick_k(X, seed):
    best_ch = -1
    for k in range(2, 51): # Test k starting from 2 to 50
        labels = KMeans(n_clusters=k, n_init='auto', random_state=seed).fit_predict(X)
        ch = calinski_harabasz_score(X, labels)
        if ch > best_ch:
            best_ch, best_k, best_labels = ch, k, labels
    return best_k, best_labels

# Function to label majority class in cluster
def majority_labels(y, cluster_labels):
    majority = y.groupby(cluster_labels).agg(lambda s: s.mode()[0])
    y_pred = majority.loc[cluster_labels].set_index(y.index)
    return majority, y_pred

### (a) Download the Anuran Calls (MFCCs) Data Set

In [21]:
filepath = '../data/Frogs_MFCCs.csv'
df = pd.read_csv(filepath)
# Split independent and target features
X = df.iloc[:, :-4]  # Grabs only MFCC data as features
y = df.iloc[:, -4:-1] # Leaves out 'RecordID'
# Split data randomly 70/30 as training/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state = 42)

### (b) Train a classifier for each label

#### (i) Research

For multi-class and mult-label classifiers, an Exact Match Ratio (ERM) method defines only exact matches (100% correct all labels) as accurately identified, while partial matches (example: 2 out of 3 features correctly identified) as misclassified. This method can be limiting as it does not account for partially correct labels and can be a harsher error evaluation metric.  
$$Exact Match Ratio, MR=1n∑i=1nI(yi=yi^)$$
Another common method is Hamming, which reports over an average how often a class label is incorrectly predicted. In contrast to ERM, hamming takes into account the prediction error and missing error, then normalizes it over total number of classes and instances. 
$$Hamming Loss = 1nL∑i=1n∑j=1LI(yji≠ŷ ji)$$

#### (ii) Train a SVM for each of the labels

In [27]:
# Train a SVM for each label
model = OneVsRestClassifier(SVC(kernel='rbf')) # Use Gaussian kernel and 1 v all classifier
params = {    
    'estimator__C': np.logspace(-1, 4, 6),     # 0.1 to 10000
    'estimator__gamma': np.logspace(-3, 1, 5)  # 0.001 to 10
}

# Train a SVM for each of the labels
print("Gaussian Model")
y_pred_gaussian, scores_gaussian = train_svm_by_label(model, params, X_train, X_test, y_train, y_test)

Fitting 10 folds for each of 30 candidates, totalling 300 fits
Fitting 10 folds for each of 30 candidates, totalling 300 fits
Fitting 10 folds for each of 30 candidates, totalling 300 fits


,Label,CV Accuracy,C,gamma
0,Family,0.992058,100.0,1.0
1,Genus,0.990070,10.0,1.0
2,Species,0.989871,10.0,1.0


,Exact Match,Hamming Loss,Hamming Score
0,0.987494,0.00772,0.99228


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [33]:
import warnings
warnings.filterwarnings('ignore')

# Fit model for each label with L1 penalty
model_l1 = Pipeline([
    ('scaler', StandardScaler()), # Scaler inside pipeline to avoid test data leakage
    ('svm', OneVsRestClassifier(
        LinearSVC(penalty='l1', dual=False, max_iter=5000))) # Use L1 penalty
])
# Use CV to find the best C
params_l1 = {'svm__estimator__C': np.logspace(-3,3,7)} # 0.001 to 1000
# Use L1 model on each label class
print("L1 Model")
y_pred_l1, scores_l1 = train_svm_by_label(model_l1, params_l1, X_train, X_test, y_train, y_test)

Fitting 10 folds for each of 7 candidates, totalling 70 fits
Fitting 10 folds for each of 7 candidates, totalling 70 fits
Fitting 10 folds for each of 7 candidates, totalling 70 fits


,Label,CV Accuracy,svm__C
0,Family,0.940629,1.0
1,Genus,0.952742,10.0
2,Species,0.960285,10.0


,Exact Match,Hamming Loss,Hamming Score
0,0.912923,0.056816,0.943184


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [34]:
# Use SMOTE to remedy class imbalance
model_smote = ImbPipeline([
    # Build the SMOTE model 
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('svm', OneVsRestClassifier(
        LinearSVC(penalty='l1', dual=False, max_iter=20000)))
])
params_smote = {'svm__estimator__C': np.logspace(-3, 2, 6)}
# Use SMOTE model on each label 
y_pred_smote, scores_smote = train_svm_by_label(model_smote, params_smote, X_train, X_test, y_train, y_test)

Fitting 10 folds for each of 6 candidates, totalling 60 fits
Fitting 10 folds for each of 6 candidates, totalling 60 fits
Fitting 10 folds for each of 6 candidates, totalling 60 fits


,Label,CV Accuracy,svm__C
0,Family,0.920973,1.0
1,Genus,0.912433,10.0
2,Species,0.957107,1.0


,Exact Match,Hamming Loss,Hamming Score
0,0.852246,0.077505,0.922495


In [35]:
# Report differences between classifiers
print("Comparison table between classifier models")
pd.concat([scores_gaussian, scores_l1, scores_smote],
          keys=['Gaussian', 'L1', 'L1 + SMOTE']).droplevel(1)

,Exact Match,Hamming Loss,Hamming Score
Gaussian,0.987494,0.007720,0.992280
L1,0.912923,0.056816,0.943184
L1 + SMOTE,0.852246,0.077505,0.922495


Question 1b.iv. Answer: The Gaussian model scored the highest. Since the actual "frog" itself is defined by its species ("Species" being a subset of "Genus" being a subset of "Family"), using Exact Match as a loss minimizer gets us .987 accuracy. Regularizing the model using L1 penalty lowers the accuracy by 7.5 points, resulting in an exact match of .913 and hamming score of .943. Since L1 uses linear boundary, and the frogs likely have similarities between family and genuses, it makes sense that the Gaussian non-linear model (more flexible) would result in a lower test error than the other linear models. This suggests that frog species are not separable by straight lines in MFCC space. The SMOTE model adjusts for rare frog classes (class imbalance) that didn't appear in the training set as proportionally, but the result was not a higher accuracy, suggesting that overall score is dominated by the common species and SMOTE's adjustment away from them lessens the usefulness of the model.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [47]:
# Use all data since this is unsupervised learning, no test/train split
X_all = df.iloc[:, :-4]
y_all = df.iloc[:, -4:-1]

# Choose a k using CH score
k0, labels0 = pick_k(X_all, seed=0)
print('Chosen k:', k0)

Chosen k: 2


Note: I used the CH score method to select a K because it gives a k fastest automatically. A scree plot requires choosing based on the visual plot given. Gap statistic and silhouette would be more time-consuming and compute costly.

### (b) Determine which family is the majority

In [49]:
# Label majority class in each cluster for family, genus, species
majority0, y_pred0 = majority_labels(y_all, labels0)
majority0   # one row per cluster: (Family, Genus, Species)

,Family,Genus,Species
0,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1,Hylidae,Hypsiboas,HypsiboasCordobae


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [44]:

# Record average and standard deviation for each run
runs = []
for seed in range(50):
    runs.append(one_run(seed))  # Runs the k-means and records best k, distance, loss, score

results = pd.DataFrame(runs, columns=['k', 'Hamming Distance', 'Hamming Loss', 'Hamming Score'])
results.agg(['mean', 'std'])

,Family,Genus,Species
0,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1,Hylidae,Hypsiboas,HypsiboasCordobae


,k,Hamming Distance,Hamming Loss,Hamming Score
mean,2.0,8.956220e-01,2.985407e-01,7.014593e-01
std,0.0,1.121495e-16,5.607473e-17,1.121495e-16


Results: CH score selected a k of 2 to use for the Monte Carlo simulation. The original data has 10 species however, so the model is already at a disadvantage to begin. The data gets divided cleanly by two distinct families of Leptodactylidae and Hylidae. The mean hamming distance for all 50 runs is .896 with a standard deviation of 0 (rounding from 1.12e^-16), meaning on average, the monte carlo model gets .896 of the 3 labels wrong.

## 3. ISLR 12.6.2

# Sketch dendogram
matrix:   1   2   3   4
       1     .3  .4  .7 
       2  .3     .5  .8
       3  .4 .5      .45  
       4  .7 .8  .45

```
a. using complete linkage (largest distance)
level 1: 1 and 2 merge at .3 --> group 1(1,2)
     1,2   3   4
1,2       .5  .8
3    .5       .45
4    .8   .45
level 2: 3 and 4 merge at .45 --> group 2(3,4)
     1,2   3,4
1,2        .8
3,4  .8
level 3: group 1(1,2) and group 2(3,4) merge at .8 --> group 3(1,2,3,4)

.8   ---|---      level 3 (group 3)
    |       |
.45 |      _|_    level 2 (group 1, group 2)
.3 _|_    |   |   level 1 (group 1,3,4)
   1 2    3   4
```

b. single linkage (shortest distance)
```
level 1: 1 and 2 merge at .3 --> group 1(1,2)
     1,2   3   4
1,2       .4  .7
3    .4       .45
4    .7   .45
level 2: group 1(1,2) and 3 merge at .4 --> group 2(1,2,3)
       1,2,3   4
1,2,3         .45
4       .45
level 3: group 1(1,2,3) and 4 merge at .45 --> group 3(1,2,3,4)

      
.45   ___|__   level 3 (group 3)
.4   _|_   |   level 2 (group 2, 4)
.3 _|_  |  |   level 1 (group 1,3,4)    
  1   2 3  4
```

c. If we cut the dendrogram obtained through complete linkage into 2 clusters, the first cluster includes obs 1 and 2 while the second cluster includes obs 3 and 4.

d. If we cut the dendrogram obtained through single linkage into 2 clusters, we would get cluster 1 with obs 1,2,3 and cluster 2 with obs 4.

e. 
```
.8   ---|----   level 3 (group 3)
     |      |
.45 _|_     |   level 2 (group 1, group 2)
.3 |   |   _|_  level 1 (group 1,3,4)
   3   4  1   2
```